# Data Cleaning & Preparation
## League of Legends Coaching Knowledge Base

---

## Background

**League of Legends (LoL)** is a competitive multiplayer game where two teams of five players each control a character called a **champion**, assigned to one of five positions called **roles** (top lane, jungle, mid lane, ADC, support). This dataset was built to power a coaching assistant that answers gameplay questions by retrieving real coaching advice from videos.

## Dataset Overview

**Source:** Two SQLite databases from a custom scraping pipeline:
- `videos.db` — 494 coaching session recordings from a professional team coach's Discord server. Discord messages were parsed to extract YouTube video links, which were then transcribed.
- `guide_test.db` — 687 YouTube educational guide videos scraped across all 172 playable champions using `yt-dlp`.

**Combined: 1,181 video rows** plus 12,000+ extracted coaching insight rows.

**`videos` table columns:**
| Column | Type | Description |
|---|---|---|
| `video_id` | TEXT (PK) | YouTube video ID |
| `video_url` | TEXT | Full YouTube URL |
| `video_title` | TEXT | Video title |
| `role` | TEXT | Lane role (top/jungle/mid/adc/support) |
| `champion` | TEXT | Champion featured in the video |
| `message_timestamp` | TEXT | Discord post timestamp (Discord source only) |
| `status` | TEXT | Pipeline stage: pending / transcribed / analyzed / no_transcript |
| `transcription` | TEXT | Full auto-generated caption text |
| `source` | TEXT | Origin: 'discord' or 'youtube_guide' |

**`insights` table columns:**
| Column | Type | Description |
|---|---|---|
| `video_id` | TEXT (FK) | Parent video |
| `insight_type` | TEXT | Category (champion_mechanics, laning_tips, matchup_advice, etc.) |
| `text` | TEXT | The extracted coaching tip |
| `source_score` | REAL | Embedding similarity to source transcript (0-1); near 0 = likely hallucinated |
| `repetition_count` | INTEGER | How many times this concept appeared in the video |

In [ ]:
import sqlite3
import pandas as pd
import re

discord_conn = sqlite3.connect('videos.db')
guide_conn   = sqlite3.connect('guide_test.db')

df_discord  = pd.read_sql('SELECT * FROM videos WHERE source = "discord"', discord_conn)
df_guides   = pd.read_sql('SELECT * FROM videos WHERE source = "youtube_guide"', guide_conn)
df_insights = pd.read_sql('SELECT * FROM insights', guide_conn)

df_videos = pd.concat([df_discord, df_guides], ignore_index=True)

print(f'Discord videos:       {len(df_discord):,} rows')
print(f'YouTube guide videos: {len(df_guides):,} rows')
print(f'Total videos:         {len(df_videos):,} rows')
print(f'Insights:             {len(df_insights):,} rows')

---
## Raw Data Examination

Before cleaning, we inspect data types, missing values, and distributions to understand where problems exist.

In [ ]:
print('=== Data Types ===')
print(df_videos.dtypes)

print('\n=== Missing / Empty Values ===')
nulls = df_videos.isnull().sum()
empty = (df_videos == '').sum()
missing = (nulls + empty).rename('total_missing')
pct = (missing / len(df_videos) * 100).round(1).rename('pct_%')
print(pd.concat([missing, pct], axis=1)[missing > 0])

print('\n=== Status Distribution ===')
print(df_videos['status'].value_counts())

print('\n=== Insights source_score ===')
print(df_insights['source_score'].describe().round(3))

---
## Step 1 — Remove Rows with No Video URL

**Issue:** The Discord scraper parsed every message in the coaching server, not just ones with YouTube links. Text-only discussion messages were inserted as rows with empty `video_url` — 108 of 494 Discord rows (~22%) have no URL.

**Reasoning:** No URL means no video and no transcript — these rows can never contribute data. Imputing a URL is impossible. We drop them rather than using a placeholder, since a missing URL causes the transcription pipeline to fail on every run.

**Result:**

In [ ]:
before = len(df_videos)
df_videos = df_videos[df_videos['video_url'].notna() & (df_videos['video_url'] != '')].copy()
print(f'Before: {before} | After: {len(df_videos)} | Removed: {before - len(df_videos)}')

---
## Step 2 — Remove Rows with Missing Champion

**Issue:** 11 rows have no value in the `champion` column. Champion is a required field — the retrieval system filters coaching insights by champion, so any video without one can never be surfaced in a query.

**Reasoning:** The champion cannot be inferred from titles like "Coaching Session #12". Imputing `"unknown"` would pollute champion-filtered results with irrelevant advice. We drop these rows as unrecoverable.

**Result:**

In [ ]:
before = len(df_videos)
df_videos = df_videos[df_videos['champion'].notna() & (df_videos['champion'] != '')].copy()
print(f'Before: {before} | After: {len(df_videos)} | Removed: {before - len(df_videos)}')

---
## Step 3 — Exclude Videos with No Transcript

**Issue:** 109 rows have `status = 'no_transcript'` — YouTube's caption API returned nothing (captions disabled, private video, or regional block). Their `transcription` column is NULL.

**Reasoning:** Without a transcript the LLM cannot extract insights. Fabricating transcript text is not an option. We exclude these from the analysis-ready dataset but keep them in the database in case captions become available later.

**Result:**

In [ ]:
before = len(df_videos)
df_clean = df_videos[df_videos['status'] != 'no_transcript'].copy()
print(f'Before: {before} | After: {len(df_clean)} | Excluded: {before - len(df_clean)}')
print('no_transcript breakdown by source:')
print(df_videos[df_videos['status'] == 'no_transcript']['source'].value_counts())

---
## Step 4 — Remove Non-English Titles

**Issue:** YouTube's search index is global, so some results were guides in French, Spanish, Korean, etc. (e.g. `"GUIDE CHO GATH FR - DETRUIRE VOS GAMES AVEC CE DINOSAURE"`). Non-English transcripts produce garbled or empty output from an English-language LLM extraction prompt.

**Reasoning:** We detect non-English titles two ways: (1) any non-ASCII character flags accented or non-Latin scripts; (2) whole-word language codes like `FR`, `ES`, `KR` in the title. We drop rather than translate — translation introduces semantic drift into coaching advice.

**Result:**

In [ ]:
LANG_TAGS = re.compile(r'\b(FR|ES|PT|DE|KR|CN|JP|IT|RU|PL|TR|NL)\b')

def is_non_english(title):
    if pd.isnull(title): return False
    if any(ord(c) > 127 for c in title): return True
    if LANG_TAGS.search(title.upper()): return True
    return False

before = len(df_clean)
mask = df_clean['video_title'].apply(is_non_english)
print(f'Non-English detected: {mask.sum()}')
for t in df_clean[mask]['video_title'].head(3): print(f'  {t}')
df_clean = df_clean[~mask].copy()
print(f'Before: {before} | After: {len(df_clean)} | Removed: {before - len(df_clean)}')

---
## Step 5 — Remove Off-Topic Game Titles

**Issue:** Riot Games publishes several titles sharing champion names — *Legends of Runeterra* (card game), *Teamfight Tactics* (auto-battler), *Wild Rift* (mobile). A search for "Draven guide" returned `"ULTIMATE Draven Midrange Deck Guide – Spiritforged"` — a card game video unrelated to PC League of Legends.

**Reasoning:** Insights from a card game video would be attributed to the LoL champion and returned as real coaching advice — a direct data quality failure. We apply a keyword blocklist against the title. A blocklist is chosen over an allowlist because LoL guide titles are too varied to enumerate positively.

**Result:**

In [ ]:
NON_LOL = ['legends of runeterra', 'runeterra', ' lor ', 'teamfight tactics',
           ' tft ', 'wild rift', 'mobile', 'card game', 'spiritforged', 'path of champions']

def is_non_lol(title):
    if pd.isnull(title): return False
    return any(kw in title.lower() for kw in NON_LOL)

before = len(df_clean)
mask = df_clean['video_title'].apply(is_non_lol)
print(f'Off-topic titles detected: {mask.sum()}')
for t in df_clean[mask]['video_title'].head(3): print(f'  {t}')
df_clean = df_clean[~mask].copy()
print(f'Before: {before} | After: {len(df_clean)} | Removed: {before - len(df_clean)}')

---
## Step 6 — Fix `message_timestamp` Data Type

**Issue:** `message_timestamp` is stored as TEXT. Discord rows contain a real timestamp string; YouTube guide rows have an empty string. Storing as TEXT prevents time-based sorting or filtering.

**Reasoning:** We convert using `pd.to_datetime(errors='coerce')` — valid timestamps parse correctly, empty strings become `NaT` (Not a Time) without raising errors. All rows are preserved with a properly typed column.

**Result:**

In [ ]:
print(f'Before dtype: {df_clean["message_timestamp"].dtype}')
df_clean['message_timestamp'] = pd.to_datetime(df_clean['message_timestamp'], errors='coerce')
print(f'After dtype:  {df_clean["message_timestamp"].dtype}')
print(f'Valid timestamps (Discord): {df_clean["message_timestamp"].notna().sum()}')
print(f'NaT (YouTube, expected):    {df_clean["message_timestamp"].isna().sum()}')

---
## Step 7 — Filter Low-Quality Insights

**Issue:** Each insight has a `source_score` (0-1) measuring how well the insight text is grounded in its source transcript. Scores near 0 mean the LLM produced a generic or hallucinated statement with no correspondence to the actual video content.

**Reasoning:** We filter insights below `0.40` as outliers representing near-zero grounding. We choose 0.40 (not higher) to retain valid insights phrased more abstractly than the transcript wording. Above this cutoff, `source_score` continues as a soft ranking weight rather than a hard gate.

**Result:**

In [ ]:
before = len(df_insights)
print(f'source_score before:\n{df_insights["source_score"].describe().round(3)}')
df_insights_clean = df_insights[df_insights['source_score'] >= 0.40].copy()
print(f'\nBefore: {before:,} | After: {len(df_insights_clean):,} | Removed: {before - len(df_insights_clean):,}')

---
## Final State — Clean Dataset

In [ ]:
print('=== Videos — Final ===')
print(f'Rows: {len(df_clean):,}')
print(df_clean['source'].value_counts().to_string())

print('\n=== Insights — Final ===')
print(f'Rows: {len(df_insights_clean):,}')
print(df_insights_clean['insight_type'].value_counts().to_string())

print('\n=== Cleaning Summary ===')
steps = [
    ('1', 'No video URL',              'videos',   'Drop rows'),
    ('2', 'Missing champion name',     'videos',   'Drop rows'),
    ('3', 'No transcript available',   'videos',   'Exclude from analysis'),
    ('4', 'Non-English titles',        'videos',   'Drop rows'),
    ('5', 'Off-topic game titles',     'videos',   'Drop rows'),
    ('6', 'message_timestamp as TEXT', 'videos',   'Type conversion'),
    ('7', 'Low source_score (<0.40)',  'insights', 'Filter rows'),
]
print(pd.DataFrame(steps, columns=['#', 'Issue', 'Table', 'Action']).to_string(index=False))